# Chapter 6 — Evaluating a model and measuring uncertainty · Solution with commentary

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch06_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each answer starts with the reasoning on paper, then shows its equivalent in Python. At each calculation, the comments point out which set is used.

**Contents.** Data · The calculations of lessons 6.1 to 6.6 · Exercises 6.1 to 6.6

**The protocol, throughout the notebook:** fit on the **training** set, choose on the **validation** set, evaluate **only once** on the **test** set. For an A/B test: compare two **randomly assigned** groups, and give the difference **with its confidence interval**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Book colors: one color per set
SET_TRAIN, SET_VAL, SET_TEST = '#2e7d32', '#b84d00', '#1565c0'   # training (green), validation (dark orange), test (blue)
SPAM, LEGIT = '#d62728', '#b9b6ae'
VERSION_A, VERSION_B, RESIDUAL, THRESHOLD = '#757575', '#00897b', '#c2185b', '#8e24aa'   # A/B test: A gray, B teal


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def mean_loss(labels, X, w, b):
    """Mean log loss (equations 5.6 and 5.7)."""
    p = np.clip(sigmoid(X @ np.asarray(w, dtype=float) + b), 1e-12, 1 - 1e-12)
    y = np.asarray(labels, dtype=float)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))


FEATURE_NAMES = ['links', 'suspicious_words', 'length', 'capitals', 'hour', 'attachments', 'size', 'recipients']


def make_messages(seed=77):
    """Fictitious messages (exercise 6.4 and lesson 6.4): 40 groups, spam campaigns (even groups) or
    legitimate senders (odd groups). The messages of a group are almost identical. Only the first two
    features depend on the label; the eight are in very different units."""
    rng = np.random.default_rng(seed)
    units = np.array([2, 1.5, 30, 0.2, 5, 1, 50, 3])
    offsets = np.array([3, 2, 200, 1, 12, 0, 500, 10])
    rows = []
    for g in range(40):
        spam = g % 2 == 0
        size = rng.integers(4, 15)
        base = rng.normal(0, 1, 8)
        base[:2] += 1.0 if spam else -1.0
        for _ in range(size):
            rows.append([g, int(spam)] + list((base + rng.normal(0, 0.3, 8)) * units + offsets))
    return pd.DataFrame(rows, columns=['group', 'spam'] + FEATURE_NAMES)


def fit(X, y, lam, l1=False, step_k=0.1, n_steps=3000):
    """Gradient descent on the penalized loss: L2 (equation 6.2) or L1 (equation 6.3). b is not penalized."""
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(n_steps):
        gap = sigmoid(X @ w + b) - y
        gw, gb = X.T @ gap / len(y), gap.mean()
        if l1:   # gradient step, then move each weight toward 0 by at most pas × λ (without changing its sign)
            w = w - step_k * gw
            w = np.sign(w) * np.maximum(np.abs(w) - step_k * lam, 0)
        else:    # the derivative of λ × sum of the w² is 2λw
            w = w - step_k * (gw + 2 * lam * w)
        b -= step_k * gb
    return w, b

## Data

The identifiers D6.1 to D6.5, D6.E1, D6.E2, D6.E3, D6.E5 and D5.1 are those of the book and of the CSV files.

In [ ]:
# Chapter 6 data, copied from the book's CSV files (do not modify)

# D6.1 — the association's three sets: messages, spams, and filter errors for a threshold
subsets = pd.DataFrame([
    ('entrainement', 600, 120, 0.7, 30, 3),
    ('validation', 200, 40, 0.3, 4, 14),
    ('validation', 200, 40, 0.5, 8, 6),
    ('validation', 200, 40, 0.7, 15, 2),
    ('test', 250, 50, 0.7, 20, 4)
], columns=['subset', 'messages', 'spams', 'threshold', 'fn', 'fp'])

# D6.2 — two splits of the same messages: the test set, at threshold 0.7
leak = pd.DataFrame([
    ('aleatoire', 250, 50, 25, 1, 10, 4),
    ('par_campagne', 250, 50, 0, 0, 20, 4)
], columns=['split', 'messages', 'spams', 'copies_seen', 'missed_copies', 'missed_new', 'fp'])

# D6.3 — six new messages (validation for lesson 6.3)
six = pd.DataFrame([
    ('R1', 'spam', 4, 2),
    ('R2', 'legitime', 1, 3),
    ('R3', 'spam', 3, 4),
    ('R4', 'legitime', 0, 0),
    ('R5', 'spam', 5, 3),
    ('R6', 'legitime', 2, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

# D5.1 — the twelve messages of chapter 5 (training for lesson 6.3)
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

# D6.E2 — the media library (exercise 6.2): validation and test sets
media_library = pd.DataFrame([
    ('validation', 150, 30, 0.4, 3, 18),
    ('validation', 150, 30, 0.5, 6, 9),
    ('validation', 150, 30, 0.6, 10, 5),
    ('validation', 150, 30, 0.7, 15, 3),
    ('test', 240, 48, 0.4, 6, 26),
    ('test', 240, 48, 0.5, 10, 14),
    ('test', 240, 48, 0.6, 17, 9),
    ('test', 240, 48, 0.7, 22, 3)
], columns=['subset', 'messages', 'spams', 'threshold', 'fn', 'fp'])

# D6.4 — the association's three folds (lesson 6.4): links (mean, standard deviation) and validation losses
folds = pd.DataFrame([
    (1, 270, 54, 2.10, 1.60, 0.29, 0.30),
    (2, 264, 52, 2.40, 1.90, 0.38, 0.33),
    (3, 266, 54, 2.20, 1.70, 0.33, 0.31)
], columns=['fold', 'messages', 'spams', 'mean_links', 'std_links', 'loss_l001', 'loss_l01'])

# D6.5 — the bookstore's A/B test (lessons 6.5 and 6.6)
bookshop = pd.DataFrame([
    ('A', 'current newsletter', 2500, 200),
    ('B', 'free delivery', 2400, 240)
], columns=['version', 'description', 'customers', 'orders'])

# D6.E1 — the media library's cross-validation (exercise 6.1): loss of each fold for five λ
folds_lambda = pd.DataFrame([
    (1, 0.001, 0.52),
    (1, 0.01, 0.44),
    (1, 0.1, 0.36),
    (1, 1, 0.35),
    (1, 10, 0.52),
    (2, 0.001, 0.40),
    (2, 0.01, 0.37),
    (2, 0.1, 0.34),
    (2, 1, 0.38),
    (2, 10, 0.55),
    (3, 0.001, 0.61),
    (3, 0.01, 0.49),
    (3, 0.1, 0.39),
    (3, 1, 0.37),
    (3, 10, 0.51)
], columns=['fold', 'lambda', 'loss'])

# D6.E3 — how four groups are spread over the volunteer's folds (exercise 6.3)
folds_of_groups = pd.DataFrame([
    ('Montres', 'spam campaign', 9, 8, 9),
    ('Colis en attente', 'spam campaign', 5, 6, 4),
    ('Loterie', 'spam campaign', 7, 7, 8),
    ('Lettre de la mairie', 'legitimate sender', 4, 3, 5)
], columns=['group', 'kind', 'fold_1', 'fold_2', 'fold_3'])

# D6.E5 — the theater's A/B test (exercise 6.5)
theater = pd.DataFrame([
    ('A', 'usual letter', 1800, 126),
    ('B', 'letter with video', 1750, 147)
], columns=['version', 'description', 'audience', 'bookings'])

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `message` | message identifier (M1…, R1…) | — | — |
| `label` | true label: spam or legitimate | — | `etiquette` |
| `links` | number of links in the message | links | `liens` |
| `suspicious_words` | number of suspicious words in the message | words | `mots_suspects` |
| `messages` | number of messages | messages | — |
| `spams` | number of spam messages | messages | — |
| `subset` | set: training, validation or test | — | `lot` |
| `threshold` | decision threshold | — | `seuil` |
| `campaign` | mailing campaign of the message | — | `campagne` |
| `group` | group of messages (a whole campaign or one sender), kept whole within one fold | — | `groupe` |
| `kind` | kind of group: spam campaign or legitimate sender | — | `nature` |
| `fold` | cross-validation fold number | — | `pli` |
| `fold_1` | D6.E3: number of messages of the group in fold 1 | messages | `pli_1` |
| `fold_2` | D6.E3: number of messages of the group in fold 2 | messages | `pli_2` |
| `fold_3` | D6.E3: number of messages of the group in fold 3 | messages | `pli_3` |
| `mean_links` | D6.4: mean number of links over the training messages of the round (the other two folds) | links | `moyenne_liens` |
| `std_links` | D6.4: standard deviation of the links over the training messages of the round (the other two folds) | links | `ecart_type_liens` |
| `description` | description of the tested version | — | — |
| `audience` | number of audience members | people | `spectateurs` |
| `bookings` | number of bookings | bookings | `reservations` |
| `customers` | number of customers exposed to the version | customers | `clients` |
| `orders` | number of orders | orders | `commandes` |
| `version` | campaign version (A or B) | — | — |
| `twelve` | the twelve messages of chapter 5 | — | — |
| `loss_l001` | D6.4: validation loss of the fold, with λ = 0.01 | — | `perte_l001` |
| `loss_l01` | D6.4: validation loss of the fold, with λ = 0.1 | — | `perte_l01` |
| `fn` | number of false negatives (missed spam) | messages | — |
| `fp` | number of false positives (legitimate messages flagged) | messages | — |
| `split` | D6.2: split of the test set (at random or by campaign) | — | `decoupage` |
| `copies_seen` | D6.2: test spam copied from a campaign seen in training | messages | `copies_vues` |
| `missed_copies` | D6.2: copies missed by the filter | messages | `manques_copies` |
| `missed_new` | D6.2: new spam missed by the filter | messages | `manques_nouveaux` |
| `lambda` | D6.E1: penalty strength λ | — | — |
| `loss` | D6.E1: validation loss of the fold for this λ | — | `perte` |

## The calculations of the lessons, in Python

These cells redo the calculations of lessons 6.1 to 6.6.

In [ ]:
# Lesson 6.1 — cost per message (equation 6.1), k = 3 for the association
k = 3
t = subsets.copy()
t['cost'] = t['fn'] + k * t['fp']
t['cost per message'] = (t['cost'] / t['messages']).round(3)    # divide by the size OF THE SET
print(t.to_string(index=False))
val = t[t['subset'] == 'validation']
chosen = val.loc[val['cost per message'].idxmin(), 'threshold']
print('threshold chosen on validation:', chosen)
test = t[(t['subset'] == 'test') & (t['threshold'] == chosen)].iloc[0]
print(f"test, only once: cost {test['cost']} for {test['messages']} messages = {test['cost per message']:.3f} per message")

a7 = t[t['threshold'] == 0.7]
colors = {'entrainement': SET_TRAIN, 'validation': SET_VAL, 'test': SET_TEST}
plt.figure(figsize=(5, 2.8))
plt.bar(a7['subset'], a7['cost per message'], color=[colors[l] for l in a7['subset']])
plt.ylabel('Cost per message (threshold 0.7)'); plt.show()

In [ ]:
# Lesson 6.2 — data leakage: recall and cost per message of the two splits
for _, l in leak.iterrows():
    missed = l['missed_copies'] + l['missed_new']
    new_spams = l['spams'] - l['copies_seen']
    print(f"{l['split']:>12} : recall = {(l['spams'] - missed) / l['spams']:.2f}, "
          f"cost per message = {(missed + 3 * l['fp']) / l['messages']:.3f}, "
          f"share of new spams missed = {l['missed_new']} / {new_spams} = {l['missed_new'] / new_spams:.2f}")

# Split the campaigns, not the messages: a small demonstration with 40 campaigns of varying size
rng = np.random.default_rng(6)
campaign = np.repeat(np.arange(40), rng.integers(5, 40, size=40))          # campaign number of each spam
at_random = rng.random(campaign.size) < 0.25                              # 25% of the MESSAGES in the test
test_campaigns = rng.choice(40, size=10, replace=False)                         # 25% of the CAMPAIGNS in the test
by_campaign = np.isin(campaign, test_campaigns)
for name, test_ in (('at random', at_random), ('by campaign', by_campaign)):
    shared = len(set(campaign[test_]) & set(campaign[~test_]))
    print(f'{name:>12} : {test_.sum()} spams in the test, {shared} campaigns present both in the test and elsewhere')

In [ ]:
# Lesson 6.3 — overfitting: filters A and B on training (twelve messages) and validation (six)
X12, y12 = twelve[['links', 'suspicious_words']].to_numpy(float), (twelve['label'] == 'spam').to_numpy(int)
X6, y6 = six[['links', 'suspicious_words']].to_numpy(float), (six['label'] == 'spam').to_numpy(int)
filters = {'A': ((1, 0.5), -4), 'B': ((-0.54, 4.39), -9.17)}
for name, (w, b) in filters.items():
    p6 = sigmoid(X6 @ np.array(w) + b)
    losses6 = -(y6 * np.log(p6) + (1 - y6) * np.log(1 - p6))
    print(f'filter {name}: training {mean_loss(y12, X12, w, b):.2f}, validation {mean_loss(y6, X6, w, b):.2f} '
          f'| losses of the six: {np.round(losses6, 2)}')

# Penalized L2 loss (equation 6.2), λ = 0.05; the constant b is not penalized
lam = 0.05
for name, (w, b) in filters.items():
    penalty = lam * np.sum(np.square(w))
    print(f'filter {name}: sum of squared weights = {np.sum(np.square(w)):.4f}, penalty = {penalty:.4f}, '
          f'penalized loss = {mean_loss(y12, X12, w, b) + penalty:.2f}')

# The penalty depends on the units: links counted in tens, same predictions, different penalty
X12_tens = X12 / np.array([10, 1])
w_tens = (10, 0.5)
print('same probabilities:', np.allclose(sigmoid(X12_tens @ np.array(w_tens) - 4), sigmoid(X12 @ np.array([1, 0.5]) - 4)),
      f'| penalty in units: {lam * (1 + 0.25):.4f}; in tens: {lam * (100 + 0.25):.4f}')

In [ ]:
# Lesson 6.4 — standardization in each round: mean and standard deviation of the links over the TWO OTHER folds
for k in range(3):
    others = folds[folds['fold'] != k + 1]
    n = others['messages'].sum()
    m = (others['messages'] * others['mean_links']).sum() / n
    # standard deviation of two folds combined (division by n): mean of the squares minus the square of the mean
    v = (others['messages'] * (others['std_links'] ** 2 + others['mean_links'] ** 2)).sum() / n - m ** 2
    print(f'round {k + 1}: validation = fold {k + 1}, training = {n} messages, mean {m:.2f}, standard deviation {np.sqrt(v):.2f}')
print(f'message with 6 links in fold 3, in round 3: (6 - 2.25) / 1.76 = {(6 - 2.25) / 1.76:.2f}')

# Cross-validation score (equation 6.4) of the two values of λ
for col, lam in (('loss_l001', 0.01), ('loss_l01', 0.1)):
    print(f'λ = {lam}: losses of the folds {list(folds[col])}, CV = {folds[col].mean():.3f}')

In [ ]:
# Lesson 6.4 (end) — L1 and L2 on fictitious messages with eight features (the same as in exercise 6.4)
msgs = make_messages()
X8 = msgs[FEATURE_NAMES].to_numpy(float)
X8 = (X8 - X8.mean(0)) / X8.std(0)        # here, standardization over all messages: just a demonstration
y8 = msgs['spam'].to_numpy(float)
w_l2, _ = fit(X8, y8, 0.1)
w_l1, _ = fit(X8, y8, 0.1, l1=True)
print('L2 weights:', np.round(w_l2, 2))
print('L1 weights:', np.round(w_l1, 2))
print(f'zero weights: L2 {int(np.sum(np.abs(w_l2) < 1e-9))}, L1 {int(np.sum(np.abs(w_l1) < 1e-9))} '
      '(the first two features are the only ones related to the label)')

In [ ]:
# Lessons 6.5 and 6.6 — the bookstore's A/B test
a, b_ = bookshop.iloc[0], bookshop.iloc[1]
pA, pB = a['orders'] / a['customers'], b_['orders'] / b_['customers']
gap = pB - pA
print(f'rate A = {100 * pA:.1f}%, rate B = {100 * pB:.1f}%, difference = {100 * gap:.1f} points, '
      f'relative change = {100 * gap / pA:.0f}%')
seA, seB = np.sqrt(pA * (1 - pA) / a['customers']), np.sqrt(pB * (1 - pB) / b_['customers'])
se = np.sqrt(seA ** 2 + seB ** 2)                          # equation 6.6: we add the SQUARES
low, high = gap - 1.96 * se, gap + 1.96 * se           # equation 6.7
print(f'SE of A = {100 * seA:.2f} point, SE of B = {100 * seB:.2f} point, SE of the difference = {100 * se:.2f} point')
print(f'95% confidence interval: from {100 * low:.1f} to {100 * high:.1f} points')
print(f'for 40,000 subscribers: {40000 * gap:.0f} more orders (from {40000 * low:.0f} to {40000 * high:.0f})')

plt.figure(figsize=(6, 1.8))
plt.axvline(0, color='black', linewidth=2.5, label='0: no effect')
plt.axvline(1, color=THRESHOLD, linestyle='--', label='break-even threshold: 1 point')
plt.plot([100 * low, 100 * high], [0, 0], color=RESIDUAL, linewidth=4, label='95% interval')
plt.scatter([100 * gap], [0], marker='D', s=70, color=RESIDUAL, zorder=3)
plt.yticks([]); plt.xlim(-1, 5); plt.xlabel('Effect of version B (percentage points)')
plt.legend(fontsize=7, loc='upper right'); plt.show()

# Lesson 6.6 — the test's cost per message (lesson 6.1) is a mean: same reasoning
costs = np.array([1] * 20 + [3] * 4 + [0] * 226)            # 20 missed spams, 4 false alarms, 226 at no cost
se_c = costs.std() / np.sqrt(len(costs))                     # standard deviation (division by n) / square root of n
print(f'cost per message = {costs.mean():.3f}, standard deviation = {costs.std():.3f}, SE = {se_c:.3f}, '
      f'interval: from {costs.mean() - 1.96 * se_c:.2f} to {costs.mean() + 1.96 * se_c:.2f}')

## Exercise 6.1 — Reading a graph: three folds, five values of λ

*Concepts: cross-validation (lesson 6.4, equation 6.4), overfitting and penalty (lesson 6.3).* Data D6.E1: the validation loss of each fold of the media library, for five values of $\lambda$.

### a) to e) Solution with commentary

**a)** Fold 2 for $\lambda = 0.1$: 0.34. Fold 3 for $\lambda = 0.001$: 0.61.

**b)** The means of the three folds are about 0.51, 0.43, 0.36, 0.37 and 0.53. The smallest is that of $\lambda = 0.1$ (0.363). On its own, fold 1 would have chosen $\lambda = 1$ (0.35), fold 2 $\lambda = 0.1$ (0.34) and fold 3 $\lambda = 1$ (0.37): two folds out of three prefer a value other than the mean's.

**c)** For $\lambda = 0.001$, the penalty holds back almost nothing: the filter memorizes the quirks of the training folds, like filter B in lesson 6.3, and is confidently wrong on the validation fold. For $\lambda = 10$, the penalty crushes all the weights toward 0: the filter gives almost the same probability to every message and no longer tells spams apart. In between, the loss goes through a dip.

**d)** The means for $\lambda = 0.1$ and $\lambda = 1$ differ by about 0.003. For the same $\lambda$, the folds differ much more: from 0.34 to 0.39 for $\lambda = 0.1$. So the choice between these two values is not solid: given the data, they are as good as each other. On the other hand, ruling out 0.001 and 10 is a solid choice.

**e)** The 0.36 is the cross-validation score that was used to **choose** $\lambda$: it slightly flatters the chosen setting. The media library must first fit the final model on all its folds with the chosen $\lambda$, settle its threshold, then measure the loss **only once** on a test set kept aside. That is the number it can announce.

In [ ]:
means = folds_lambda.groupby('lambda')['loss'].mean()     # CV(λ): mean of the three folds
lam_chosen = means.idxmin()

In [ ]:
# Check of a) and b)
if means is None or lam_chosen is None:
    print('not yet completed. Complete the two lines.')
else:
    print('means:', means.round(3).to_dict(), '→',
          'correct' if np.allclose(means.values, [0.51, 1.3 / 3, 1.09 / 3, 1.1 / 3, 1.58 / 3]) else 'check again')
    print('chosen λ:', lam_chosen, '→', 'correct' if lam_chosen == 0.1 else 'check again')
    print('choice of each fold on its own:', folds_lambda.loc[folds_lambda.groupby('fold')['loss'].idxmin(), ['fold', 'lambda']].values.tolist())
    for k, (style, marker) in enumerate((('-', 'o'), ('--', 's'), (':', '^'))):
        d = folds_lambda[folds_lambda['fold'] == k + 1]
        plt.plot(d['lambda'], d['loss'], style, marker=marker, color=SET_VAL, label=f'fold {k + 1}')
    plt.plot(means.index, means.values, 'D-', color='black', linewidth=2.5, label='mean')
    plt.xscale('log'); plt.xlabel('λ'); plt.ylabel('Validation loss'); plt.legend(); plt.grid(alpha=0.4); plt.show()

## Exercise 6.2 — Choosing, then evaluating a threshold: the media library

*Concepts: three sets (lesson 6.1), cost per message (equation 6.1), cost of a threshold (equation 5.9).* For the media library, one false alarm costs as much as two missed spams: $k = 2$. Data D6.E2. First do questions a) to e) on paper, **without looking at the test rows before question b)**.

### a) to e) Solution with commentary

**a)** On the **validation** set (150 messages), with $FN + 2 \times FP$:

| Threshold | Cost | Cost per message |
|:--|--:|--:|
| 0.4 | $3 + 2 \times 18 = 39$ | $39 / 150 = 0.260$ |
| 0.5 | $6 + 2 \times 9 = 24$ | $24 / 150 = 0.160$ |
| 0.6 | $10 + 2 \times 5 = 20$ | $20 / 150 \approx 0.133$ |
| 0.7 | $15 + 2 \times 3 = 21$ | $21 / 150 = 0.140$ |

Threshold **0.6** costs the least on the validation set. We keep it.

**b)** On the **test** set (240 messages), at threshold 0.6 only: $17 + 2 \times 9 = 35$, then $35 / 240 \approx 0.146$ per message. This is the number to announce.

**c)** The two sets are not the same size: 150 and 240 messages. The totals, 20 and 35, would suggest that the filter costs 75% more on the test set. Per message, the gap is only from 0.133 to 0.146, about 9% more. A total grows with the number of messages; a cost per message does not.

**d)** Yes. The threshold was chosen because it did best on those particular 150 messages, with their share of luck. Its validation performance is therefore slightly flattering. The test set, which was not used to decide anything, does not benefit from that luck.

**e)** First mistake: he **chooses the threshold on the test set**, which becomes a second validation set. Second mistake: he announces the **best** of the four test costs, 0.117, which is bound to be flattering: it is a minimum, not a measurement. He should announce 0.146 per message, at the threshold 0.6 chosen on the validation set. If he really wants to compare 0.6 and 0.7, he needs a larger validation set or cross-validation, then a new test set. Note, by the way, that on the validation set 0.6 and 0.7 differ by only one unit of cost over 150 messages: the choice between them is fragile.

In [ ]:
def cost_per_message(fn, fp, n, k):
    return (fn + k * fp) / n                    # divide by the size of the set


def threshold_choice(validation_table, k):
    """Returns the threshold whose cost per message is the smallest on the validation set."""
    costs = [cost_per_message(l.fn, l.fp, l.messages, k) for l in validation_table.itertuples()]
    return validation_table['threshold'].iloc[int(np.argmin(costs))]   # the test is not looked at here

In [ ]:
# Check of a) and b), then the protocol
k_med = 2
valid = media_library[media_library['subset'] == 'validation']
test_med = media_library[media_library['subset'] == 'test']
if cost_per_message(3, 18, 150, 2) is None or threshold_choice(valid, k_med) is None:
    print('not yet completed. Complete the functions.')
else:
    cv = [cost_per_message(l.fn, l.fp, l.messages, k_med) for l in valid.itertuples()]
    print('validation, cost per message:', np.round(cv, 3), '→',
          'correct' if np.allclose(cv, [39 / 150, 24 / 150, 20 / 150, 21 / 150]) else 'check again')
    s = threshold_choice(valid, k_med)
    print(f'threshold chosen on validation: {s} →', 'correct' if s == 0.6 else 'check again')
    l = test_med[test_med['threshold'] == s].iloc[0]                    # ONLY ONE row of the test
    ct = cost_per_message(l.fn, l.fp, l.messages, k_med)
    print(f'test, at threshold {s} only: {ct:.3f} per message →', 'correct' if abs(ct - 35 / 240) < 1e-9 else 'check again')
    # e) What the colleague does: he looks at ALL the rows of the test and keeps the best one
    ct_all = [cost_per_message(r.fn, r.fp, r.messages, k_med) for r in test_med.itertuples()]
    s_coll = test_med['threshold'].iloc[int(np.argmin(ct_all))]
    print(f'what the colleague announces: threshold {s_coll}, {min(ct_all):.3f} per message (chosen on the test: too optimistic);',
          f'honest announcement: threshold {s}, {ct:.3f} per message →',
          'correct' if s_coll == 0.7 and abs(min(ct_all) - 28 / 240) < 1e-9 else 'check again')
    plt.figure(figsize=(5.5, 3))
    plt.plot(valid['threshold'], cv, 'o-', color=SET_VAL, label='validation (used to choose)')
    plt.plot(test_med['threshold'], ct_all, 's--', color=SET_TEST, alpha=0.5, label='test (not to be looked at when choosing)')
    plt.scatter([s], [ct], s=120, color=SET_TEST, zorder=3, label='test, at the chosen threshold')
    plt.xlabel('Threshold'); plt.ylabel('Cost per message (k = 2)'); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### f) Solution with commentary

The media library can announce: "At threshold 0.6, chosen on the validation set, the filter costs about 0.146 per message on a test set of 240 messages that had not been used for anything else." The graph shows why the colleague's announcement is too good: by choosing the lowest point of the test curve, he turns the test set into a validation set. His 0.117 is no longer a measurement of the cost to come. Finally, 240 messages do not give an exact cost: lesson 6.6 teaches how to say by how much it can vary.

## Exercise 6.3 — Interpreting: a leak between folds

*Concepts: data leakage (lesson 6.2), group cross-validation and standardization in each round (lesson 6.4).* Data D6.E3: where the messages of four groups ended up in the volunteer's folds.

### a) to d) Solution with commentary

**a)** In round 1, folds 2 and 3 form the training set: the filter has seen $8 + 9 = 17$ copies of the “Montres” (watches) campaign. It finds 9 of them in fold 1, in validation.

**b)** **Step 2**: the fold is drawn message by message. So every campaign is split across the folds (all four groups in the table are). In each round, the filter has seen in training copies almost identical to the validation messages: it recognizes the campaign instead of judging a new message. **Step 1**: the mean and standard deviation are computed on the 800 messages, so including the messages of each validation fold. These two numbers pass a little information from validation into training. This second leak is smaller, but of the same kind.

**c)** The test loss, 0.34, best describes the messages to come: the test set was formed by campaign and was used only once. The cross-validation score, 0.11, is too good because the validation contained copies already seen. Steps 4 and 5 are correct in themselves: choose $\lambda$ by cross-validation, then open the test set only once. But step 4 works on distorted scores: the leak rewards filters that memorize, so the chosen $\lambda$ may itself be too small.

**d)** The groups have 26, 15, 22 and 12 messages, 75 in all; the ideal would be 25 per fold. One possible distribution: fold 1 = Montres (26), fold 2 = Loterie (22), fold 3 = Colis en attente and Lettre de la mairie (27). Step 1 rewritten: "In each round, compute the mean and standard deviation of each feature on the two training folds only, then use these two numbers to standardize the training messages and those of the validation fold."

In [ ]:
def split_groups(allocation):
    """allocation: a table with a 'group' column and the columns 'fold_1', 'fold_2', 'fold_3'
    (number of messages of the group in each fold). Returns the list of split groups."""
    n_folds = (allocation[['fold_1', 'fold_2', 'fold_3']] > 0).sum(axis=1)   # in how many folds is the group?
    return list(allocation.loc[n_folds > 1, 'group'])

In [ ]:
# Check of b) and d)
proposal = pd.DataFrame([('Montres', 26, 0, 0), ('Colis en attente', 0, 0, 15),
                            ('Loterie', 0, 22, 0), ('Lettre de la mairie', 0, 0, 12)],
                           columns=['group', 'fold_1', 'fold_2', 'fold_3'])
if split_groups(folds_of_groups) is None:
    print('not yet completed. Complete the function.')
else:
    print('folds of the volunteer, split groups:', split_groups(folds_of_groups), '→',
          'correct' if len(split_groups(folds_of_groups)) == 4 else 'check again')
    print('corrected distribution, split groups:', split_groups(proposal), '| fold sizes:',
          proposal[['fold_1', 'fold_2', 'fold_3']].sum().tolist(), '→',
          'correct' if split_groups(proposal) == [] else 'check again')
    print('round 1: copies of “Montres” seen in training =', int(folds_of_groups.loc[0, ['fold_2', 'fold_3']].sum()),
          ', in validation =', int(folds_of_groups.loc[0, 'fold_1']))

## Exercise 6.4 — Cross-validation by campaign

*Concepts: cross-validation (lesson 6.4, equation 6.4), L2 penalty (equation 6.2), standardization (equation 2.7), leakage (lesson 6.2).* The messages are made by `make_messages()` (import cell). Groups 30 to 39 form the test set: we touch it only in question e).

In [ ]:
# The messages of the exercise: 30 groups for cross-validation, 10 groups set aside for the test
msgs = make_messages()
test_ = msgs['group'] >= 30
X_cv, y_cv, g_cv = msgs.loc[~test_, FEATURE_NAMES].to_numpy(float), msgs.loc[~test_, 'spam'].to_numpy(float), msgs.loc[~test_, 'group'].to_numpy()
X_test, y_test = msgs.loc[test_, FEATURE_NAMES].to_numpy(float), msgs.loc[test_, 'spam'].to_numpy(float)
print(f'{len(msgs)} messages, {msgs.group.nunique()} groups; cross-validation: {len(y_cv)} messages; test: {len(y_test)} messages')
print(msgs[FEATURE_NAMES].describe().loc[['mean', 'std']].round(1))     # very different units

### a) Solution with commentary

We expect it to **fall, then rise again**. Without a penalty, the filter can give large weights to the six features unrelated to the label, which vary by chance from one group to another: it overfits. With too strong a penalty, all the weights are crushed.

In [ ]:
def group_folds(groups, K):
    """Fold number (0 to K - 1) of each message: the groups, sorted in increasing order, are dealt
    in turn into the folds; all the messages of a group get the same fold."""
    ranks = {g: i % K for i, g in enumerate(np.unique(groups))}
    return np.array([ranks[g] for g in groups])


def standardization(X_train, X_val):
    """Standardizes X_train and X_val with the mean and standard deviation of X_train ONLY."""
    m, s = X_train.mean(axis=0), X_train.std(axis=0)
    return (X_train - m) / s, (X_val - m) / s


def cv_score(lam, folds_, K=3):
    """Equation 6.4: mean, over the K rounds, of the loss WITHOUT penalty on the validation fold."""
    losses = []
    for k in range(K):
        train, val = folds_ != k, folds_ == k
        Xe, Xv = standardization(X_cv[train], X_cv[val])          # numbers computed within the round, on training
        w, b = fit(Xe, y_cv[train], lam)
        losses.append(mean_loss(y_cv[val], Xv, w, b))     # we judge the predictions, without penalty
    return float(np.mean(losses))

In [ ]:
# Check of b) and c)
LAMBDAS = [0, 0.001, 0.01, 0.1, 1]
p_g = group_folds(g_cv, 3)
if p_g is None or standardization(X_cv, X_cv)[0] is None or cv_score(0.1, p_g) is None:
    print('not yet completed. Complete the three functions.')
else:
    splits = sum(len(set(p_g[g_cv == g])) > 1 for g in np.unique(g_cv))
    print(f'folds by group: {np.bincount(p_g).tolist()} messages, {splits} split group(s) →',
          'correct' if splits == 0 and set(p_g) == {0, 1, 2} else 'check again')
    Xe, Xv = standardization(X_cv[p_g != 0], X_cv[p_g == 0])
    print('standardized training: means ≈ 0 and standard deviations ≈ 1; validation: not exactly →',
          'correct' if np.allclose(Xe.mean(0), 0) and np.allclose(Xe.std(0), 1) and not np.allclose(Xv.mean(0), 0) else 'check again')
    cv_group = {lam: cv_score(lam, p_g) for lam in LAMBDAS}
    lam_cv = min(cv_group, key=cv_group.get)
    print('CV by group:', {l: round(v, 3) for l, v in cv_group.items()}, '→ chosen λ:', lam_cv, '→',
          'correct' if lam_cv == 0.01 else 'check again')

In [ ]:
# d) Folds drawn message by message (not to be done): the leakage of lesson 6.2, three times over
if p_g is not None and cv_score(0.1, p_g) is not None:
    p_m = np.random.default_rng(1).integers(0, 3, len(y_cv))
    cv_message = {lam: cv_score(lam, p_m) for lam in LAMBDAS}
    print('CV message by message:', {l: round(v, 3) for l, v in cv_message.items()})
    plt.plot(range(5), list(cv_group.values()), 'o-', color=SET_VAL, label='folds by group')
    plt.plot(range(5), list(cv_message.values()), 's--', color='gray', label='folds message by message (leakage)')
    plt.xticks(range(5), [str(l) for l in LAMBDAS]); plt.xlabel('λ'); plt.ylabel('Cross-validation score')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

    # e) Final model: standardization and weights on the 30 groups, with the chosen λ; test ONLY ONCE
    Xc, Xt = standardization(X_cv, X_test)
    w_f, b_f = fit(Xc, y_cv, lam_cv)
    print(f'test loss of the final model (λ = {lam_cv}): {mean_loss(y_test, Xt, w_f, b_f):.3f}; '
          f'CV by group: {cv_group[lam_cv]:.3f}; CV message by message: {cv_message[lam_cv]:.3f}')

### d) and e) Solution with commentary

**c)** With folds by group, the cross-validation score is about 0.547 for $\lambda = 0$, 0.493 for 0.001, **0.378 for 0.01**, 0.424 for 0.1 and 0.609 for 1. It falls, then rises again: we keep $\lambda = 0.01$. Without a penalty, the filter overfits on the six features unrelated to the label; with $\lambda = 1$, the weights are crushed too much.

**d)** With folds drawn message by message, the scores drop to 0.207, 0.206, 0.246, 0.385 and 0.598. Each group is split across the folds: in each round, the validation contains almost identical copies of training messages, and the filter recognizes them. The score is too good, especially for small $\lambda$, which reward memorizing: this split would even have picked $\lambda = 0.001$. The leak distorts both the measurement and the choice.

**e)** The final model ($\lambda = 0.01$, standardization and weights computed on the 30 groups) has a test loss of 0.356, measured only once on 10 groups never seen before. It is close to the group cross-validation score, 0.378, and far from the 0.246 of the message-by-message folds. Group cross-validation honestly announced what awaited the filter; the other was off by a third. The remaining gap between 0.356 and 0.378 is due to chance: 94 test messages is not many.

## Exercise 6.5 — Predict, then check: the theater's reminder letter

*Concepts: difference in points (equation 6.5), standard error (equation 6.6), confidence interval (equation 6.7).* Data D6.E5. The video is only worthwhile if it gains at least 0.5 points.

### a) to e) Solution with commentary

**a)** Rate A: $126 / 1{,}800 = 7.0\%$. Rate B: $147 / 1{,}750 = 8.4\%$. Difference: $8.4 - 7.0 = 1.4$ percentage points. Relative change: $1.4 / 7.0 = 20\%$.

**b)** In points: $7 \times 93 / 1{,}800 \approx 0.362$ and $8.4 \times 91.6 / 1{,}750 \approx 0.440$. $\text{SE} = \sqrt{0.362 + 0.440} = \sqrt{0.801} \approx 0.90$ point. Interval: $1.4 \pm 1.96 \times 0.90 \approx 1.4 \pm 1.75$, that is, from about **−0.35 to 3.15 points**.

**c)** *The sign.* The measured difference is positive, but zero is in the interval, and even small negative effects: the trial does not allow us to claim that the video led to more bookings. *The precision.* The interval is 3.5 points wide, two and a half times the difference itself: the effect is very poorly known. *The practical value.* The 0.5-point threshold is in the interval, just like zero and effects of 3 points: the video may be very useful or useless; the data do not decide. The colleague confuses "the interval contains zero" with "the effect is zero": a trial that is too small does not prove the absence of an effect. 1.4 points is still the most plausible value; what is missing is precision.

**d)** The standard error varies like $1 / \sqrt{n}$: four times as many audience members divide the standard error, and hence the width of the interval, by $\sqrt{4} = 2$, not by 4.

**e)** $\text{SE} = \sqrt{0.801 / 4} \approx 0.45$ point, and the interval goes from $1.4 - 0.88 \approx 0.52$ to $1.4 + 0.88 \approx 2.28$ points. Zero would be excluded, and the 0.5-point threshold only just: if the rates stayed the same, the video would be judged useful, narrowly.

In [ ]:
def interval(conv_A, n_A, conv_B, n_B):
    """Returns (difference, lower bound, upper bound), in percentage points."""
    pA, pB = conv_A / n_A, conv_B / n_B
    se = np.sqrt(pA * (1 - pA) / n_A + pB * (1 - pB) / n_B)   # equation 6.6
    return 100 * (pB - pA), 100 * (pB - pA - 1.96 * se), 100 * (pB - pA + 1.96 * se)

In [ ]:
# Check of a), b) and e)
A, B = theater.iloc[0], theater.iloc[1]
r = interval(A['bookings'], A['audience'], B['bookings'], B['audience'])
if r is None:
    print('not yet completed. Complete the function.')
else:
    print(f'actual trial: difference {r[0]:.2f} points, interval from {r[1]:.2f} to {r[2]:.2f} →',
          'correct' if abs(r[0] - 1.4) < 1e-9 and abs(r[1] + 0.35) < 0.01 and abs(r[2] - 3.15) < 0.01 else 'check again')
    r4 = interval(4 * A['bookings'], 4 * A['audience'], 4 * B['bookings'], 4 * B['audience'])
    print(f'trial four times larger: from {r4[1]:.2f} to {r4[2]:.2f}; width {r[2] - r[1]:.2f} → {r4[2] - r4[1]:.2f} →',
          'correct' if abs((r[2] - r[1]) / (r4[2] - r4[1]) - 2) < 1e-9 and r4[1] > 0.5 else 'check again')

In [ ]:
# f) 1,000 simulated trials whose true effect is 1.4 points: how many intervals contain 1.4?
rng = np.random.default_rng(65)
nA, nB, true_A, true_B = 1800, 1750, 0.07, 0.084
a = rng.binomial(nA, true_A, 1000) / nA
b = rng.binomial(nB, true_B, 1000) / nB
se_sim = np.sqrt(a * (1 - a) / nA + b * (1 - b) / nB)
contains = np.abs((b - a) - 0.014) <= 1.96 * se_sim
print(f'share of intervals that contain the true effect: {contains.mean():.1%}')
print(f'share of intervals that contain zero: {np.mean(np.abs(b - a) <= 1.96 * se_sim):.1%}')

### f) Solution with commentary

About 94% of the 1,000 simulated intervals contain the true effect, 1.4 points: this is the "95%", up to the randomness of the simulation. It describes the **method**: over a large number of trials, about 95% of the intervals built this way contain the true effect. Note also that with a true effect of 1.4 points, a good share of the intervals contain zero: a trial of this size often misses a real effect. That is one more reason not to read "contains zero" as "no effect".

## Exercise 6.6 — Challenge: how many customers, and when to look?

*Concepts: standard error and confidence interval (lesson 6.6), binomial distribution (lesson 4.5).*

### a) to c) Solution with commentary

**a)** **Sizing goal: precision.** We want the 95% interval to stray no more than $m = 1$ point from the measured difference. This is not a power calculation: we set neither a hypothesis to test nor a probability of detecting an effect. **Assumptions:** two randomly assigned groups of the same size $n$; rates assumed equal to $p_A = 8$% and $p_B = 10$%, those of the trial in lesson 6.5; approximation by the bell curve ($z = 1.96$); interval read only once, at the end.

In points, $\text{SE} = \sqrt{\big(p_A(100 - p_A) + p_B(100 - p_B)\big) / n}$, and the condition $z \times \text{SE} \le m$ gives:

$$n \ge \frac{z^2 \, \big(p_A(100 - p_A) + p_B(100 - p_B)\big)}{m^2} = \frac{1.96^2 \times (8 \times 92 + 10 \times 90)}{1^2} = 3.8416 \times 1{,}636 \approx 6{,}284.9 .$$

We round up to the next integer: **6,285 customers per group**, about 12,570 in total. These 6,285 customers correspond to an **approximate** margin of one point: approximate because the bell curve is an approximation, and because the calculation assumes rates of 8% and 10%. If the true rates differ, the size changes: with rates closer to 50%, more customers would be needed.

**b)** The interval would go from $2.0 - 1.0 = 1.0$ to $3.0$ points: it would touch the threshold without excluding it. For a half-width of 0.5 points, the standard error must be divided by 2, so $n$ multiplied by 4: about 25,140 customers per group, 50,280 in total. That is more than the 40,000 subscribers: a single mailing would not be enough; the trial would have to be repeated over two mailings, for example.

**c)** No: we expect much more than 5%, because each evening gives chance a new opportunity to cross the limit.

In [ ]:
def group_size(pA, pB, half_width):
    """pA, pB in percent; returns the smallest integer n such that 1.96 × SE ≤ half_width."""
    return int(np.ceil((1.96 / half_width) ** 2 * (pA * (100 - pA) + pB * (100 - pB))))

In [ ]:
# Check of a) and b)
n1 = group_size(8, 10, 1)
if n1 is None:
    print('not yet completed. Complete the function.')
else:
    print(f'half-width 1 point: {n1} customers per group →', 'correct' if n1 == 6285 else 'check again')
    print(f'half-width 0.5 point: {group_size(8, 10, 0.5)} customers per group, '
          f'{2 * group_size(8, 10, 0.5)} in total')

In [ ]:
# d) 2,000 trials WITHOUT any effect. Schedule: 250 customers per day and per group, 20 readings (one each evening,
#    on the cumulative customers: 250, 500… 5,000 per group). Stopping rule: first evening on which the interval excludes 0.
rng = np.random.default_rng(66)
trials, days, per_day, p = 2000, 20, 250, 0.08
cA = rng.binomial(per_day, p, size=(trials, days)).cumsum(axis=1)     # cumulative orders, group A
cB = rng.binomial(per_day, p, size=(trials, days)).cumsum(axis=1)     # group B: same rate, no effect
n = per_day * np.arange(1, days + 1)                                  # customers per group at the end of each day
pA_, pB_ = cA / n, cB / n
se_j = np.sqrt(pA_ * (1 - pA_) / n + pB_ * (1 - pB_) / n)
excludes = np.abs(pB_ - pA_) > 1.96 * se_j                                # does the evening's interval exclude zero?
end, one_evening = excludes[:, -1].mean(), excludes.any(axis=1).mean()
print(f'interval read only on day 20: {end:.1%} of the trials wrongly conclude there is an effect')
print(f'interval read every evening, stopping at the first “success”: {one_evening:.1%}')

### d) and e) Solution with commentary

Simulation conditions: 2,000 trials without any effect (rate of 8% in both groups), 250 new customers per day and per group, at most 20 readings, one each evening from day 1 to day 20, on the customers accumulated since the start (from 250 to 5,000 per group); stopping rule: we stop on the first evening when the 95% interval excludes zero, on either side, and conclude there is an effect. Read only once, on the 20th evening, the interval excludes zero in about 5% of the trials without any effect (4.6% in the simulation): this is the planned risk. With a reading every evening and the stopping rule, we wrongly conclude in about one trial in four (24.7%). This figure depends on the schedule: more readings, or readings closer together, would push it even higher. Each evening is a new opportunity for chance to push the interval away from zero; stopping at that moment means keeping the favorable opportunities and throwing away the others. The rule: **fix the size of the trial and the reading date in advance**, for example 6,300 customers per group based on a), and read the interval on that date. Looking at the figures along the way is not forbidden, but must not decide when to stop.